# 20. Controlled Stage 1 Prior-Policy Diagnostic — Facial Skincare

This independent diagnostic compares No Prior, QCHS, and All Prior while holding the Graph-Hybrid query-only backbone, Profile Sparse architecture, candidate budget, item representation, fusion weights, RRF constant, and deterministic tie-breaking rule fixed. QCHS uses the leakage-safe top-12 query-aligned training-safe prior items. All Prior retains every eligible strict pre-target event without item-level deduplication. Cold and policy-fallback cases must reproduce the No Prior candidate list exactly.

The thesis-reported archived result set is evaluated at candidate depth 700 with NDCG@5. It reports population means and paired contrasts for all cases, non-cold cases, the Strong regime, and the Stage-1 QCHS-active population, using 2,000 user-clustered bootstrap replicates. All results are descriptive and unadjusted.

This diagnostic does not re-select the canonical Stage 1 winner or modify the main pipeline. Stage-1 QCHS activity is defined from query-aligned training-safe history and is distinct from the Stage-2 QCHS-filtered-prior population.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import importlib.util
import subprocess
import sys

package_map = {
    "pyarrow": "pyarrow",
    "rank_bm25": "rank-bm25",
    "tqdm": "tqdm",
}
if "profile_sparse" == "profile_hybrid":
    package_map.update({
        "faiss": "faiss-cpu",
        "sentence_transformers": "sentence-transformers",
    })
missing_packages = [package for module, package in package_map.items() if importlib.util.find_spec(module) is None]
if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])


In [3]:
from collections import Counter, defaultdict
from pathlib import Path
import hashlib
import json
import math
import re
import time

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display
from rank_bm25 import BM25Okapi
from tqdm.auto import tqdm

if "profile_sparse" == "profile_hybrid":
    import faiss
    from sentence_transformers import SentenceTransformer

pd.set_option("display.max_columns", 200)


In [4]:
# =========================================================
# Fixed configuration: no selection or test-metric tuning
# =========================================================
CATEGORY_ID = "face"
CATEGORY_LABEL = "Facial Skincare"
PROJECT_ROOT = Path("/content/drive/MyDrive/thesis_recsys/categories/facial_skincare")

QUERY_CACHE_PATH = PROJECT_ROOT / "outputs/query_cache/face_queries.parquet"
QUERY_CONTRACT_PATH = PROJECT_ROOT / "outputs/query_summary/face_queries_config.json"
PRIOR_HISTORY_PATH = PROJECT_ROOT / "data/processed/user_sampling/face_user_prior_review_history_training.parquet"
SAMPLING_MANIFEST_PATH = PROJECT_ROOT / "data/processed/user_sampling/face_user_regime_sampling_manifest.json"
ITEM_DOCS_PATH = PROJECT_ROOT / "data/processed/items/face_item_docs.parquet"
ITEM_FACETS_PATH = PROJECT_ROOT / "data/processed/items/face_items_facets.parquet"
RETRIEVAL_ARTIFACT_MANIFEST_PATH = PROJECT_ROOT / "data/processed/items/retrieval_artifact_manifest_face.json"
STAGE1_MANIFEST_PATH = PROJECT_ROOT / "outputs/stage1_query_retrieval_selection/stage1_run_manifest_face.json"
STAGE1_WINNER_MANIFEST_PATH = PROJECT_ROOT / "outputs/stage1_query_retrieval_selection/stage1_query_only_winner_face.json"

OUTPUT_DIR = PROJECT_ROOT / "outputs/analysis/stage1_prior_policy_control"
CANDIDATES_PATH = OUTPUT_DIR / "stage1_prior_policy_candidates.parquet"
PER_CASE_PATH = OUTPUT_DIR / "stage1_prior_policy_per_case.parquet"
CONTRAST_PER_CASE_PATH = OUTPUT_DIR / "stage1_prior_policy_contrasts_per_case.parquet"
PAIRED_SUMMAR_PATH = OUTPUT_DIR / "stage1_prior_policy_paired_summary.csv"
POPULATION_RESULTS_PATH = OUTPUT_DIR / "stage1_prior_policy_population_results.csv"
PROFILE_DIAGNOSTICS_PATH = OUTPUT_DIR / "stage1_prior_policy_profile_diagnostics.parquet"
COVERAGE_PATH = OUTPUT_DIR / "stage1_prior_policy_coverage_summary.csv"
RETENTION_PATH = OUTPUT_DIR / "stage1_prior_policy_history_retention_summary.csv"
FALLBACK_IDENTITY_QC_PATH = OUTPUT_DIR / "stage1_prior_policy_fallback_identity_qc.csv"
HISTORY_EXCLUSIONS_PATH = OUTPUT_DIR / "stage1_prior_policy_history_exclusions.csv"
QC_SUMMAR_PATH = OUTPUT_DIR / "stage1_prior_policy_qc_summary.csv"
MANIFEST_PATH = OUTPUT_DIR / "stage1_prior_policy_manifest.json"

ACTIVE_QUERY_COLUMN = "query"
COMPATIBILITY_QUERY_ALIAS = None
PRIOR_HAS_TARGET_PARENT_ASIN = False
QUERY_PASSTHROUGH_COLUMNS = ['sampling_bracket']
REGIME_ORDER = ['cold', 'weak', 'moderate', 'strong']

DENSE_TEXT_COLUMN = "dense_text"
SPARSE_TEXT_COLUMN = "sparse_text"
BRAND_TEXT_COLUMN = "brand_facet_text"
PROFILE_SAFE_TEXT_COLUMN = "profile_safe_facet_text"
ITEM_EVIDENCE_SCOPE = "catalog_metadata_functional_facets_and_historical_review_signals"
PROFILE_EVIDENCE_SCOPE = "leakage_safe_strict_pre_target_prior_item_metadata_functional_and_brand_facets"

EXPECTED_BASELINE_METHOD_KEY = "graph_hybrid"
EXPECTED_BASELINE_METHOD_LABEL = "Graph-Hybrid"
ARCHITECTURE = "profile_sparse"
ARCHITECTURE_LABEL = "Profile Sparse"
POLICY_ORDER = ["no_prior", "qchs", "all_prior"]
POLICY_LABELS = {
    "no_prior": "No Prior — Graph-Hybrid",
    "qchs": "QCHS — Profile Sparse QCHS",
    "all_prior": "All Prior — Profile Sparse All Prior",
}

POOL_DEPTH = 1000
PRIMARY_K = 5
MAX_QCHS_PRIOR_ITEMS = 12
MAX_ANCHOR_PHRASES = 16
MAX_ANCHOR_PHRASES_PER_ROLE = 4
QUERY_REPEAT = 3
RRF_K = 60
FUSION_WEIGHTS = [1.0, 0.8]
ATTENTION_TEMPERATURE = 0.25
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
ITEM_EMBEDDING_BATCH_SIZE = 128
QUERY_EMBEDDING_BATCH_SIZE = 256
BOOTSTRAP_REPS = 2000
BOOTSTRAP_CONFIDENCE = 0.95
RANDOM_SEED = 42

FUNCTIONAL_ROLES = {
    "category_or_product_type", "claim_constraint", "form_texture",
    "ingredient_or_composition", "need_benefit_concern", "sensory", "target_context",
}
PROFILE_ROLES = set(FUNCTIONAL_ROLES) | {"brand"}
TOKEN_EQUIVALENCE = {'acne': 'acne', 'blemish': 'acne', 'blemishes': 'acne', 'breakout': 'acne', 'breakouts': 'acne', 'brighten': 'brightening', 'brightening': 'brightening', 'brightness': 'brightening', 'cleanser': 'cleanser', 'cleansers': 'cleanser', 'cleansing': 'cleanser', 'wash': 'cleanser', 'cream': 'moisturizer', 'creams': 'moisturizer', 'lotion': 'moisturizer', 'lotions': 'moisturizer', 'moisturize': 'moisturizer', 'moisturizer': 'moisturizer', 'moisturizers': 'moisturizer', 'moisturizing': 'moisturizer', 'dark': 'dark', 'hyperpigmentation': 'dark_spots', 'pigmentation': 'dark_spots', 'spot': 'dark_spots', 'spots': 'dark_spots', 'dry': 'dryness', 'dryness': 'dryness', 'hydrate': 'hydration', 'hydrated': 'hydration', 'hydrating': 'hydration', 'hydration': 'hydration', 'oiliness': 'oiliness', 'oily': 'oiliness', 'pore': 'pores', 'pores': 'pores', 'sensitive': 'sensitivity', 'sensitivity': 'sensitivity', 'mask': 'mask', 'masks': 'mask', 'serum': 'serum', 'serums': 'serum', 'toner': 'toner', 'toners': 'toner', 'treatment': 'treatment', 'treatments': 'treatment', 'wrinkle': 'wrinkles', 'wrinkles': 'wrinkles'}
LINGUISTIC_STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "by", "for", "from", "in", "into",
    "is", "it", "of", "on", "or", "that", "the", "this", "to", "with", "without", "you", "your",
}
GENERIC_ANCHOR_TOKENS = {'product', 'solution', 'face', 'routine', 'care', 'skin', 'item', 'solutions', 'support', 'skincare', 'items', 'products', 'facial'}
ATTENTION_STOPWORDS = {'in', 'of', 'at', 'routine', 'an', 'under', 'to', 'be', 'that', 'are', 'helps', 'formula', 'into', 'its', 'skin', 'blend', 'by', 'over', 'skincare', 'support', 'the', 'these', 'a', 'on', 'face', 'is', 'complex', 'and', 'this', 'care', 'it', 'those', 'help', 'supports', 'without', 'facial', 'product', 'solution', 'boost', 'promotes', 'with', 'from', 'promote', 'as', 'or', 'for'}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Diagnostic output:", OUTPUT_DIR)
print("Fixed architecture:", ARCHITECTURE_LABEL, FUSION_WEIGHTS)


Diagnostic output: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/analysis/stage1_prior_policy_control
Fixed architecture: Profile Sparse [1.0, 0.8]


In [5]:
# =========================================================
# Shared helpers copied from the canonical Notebook 08 logic
# =========================================================
def normalize_space(value):
    if value is None or pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value).replace("\n", " ").replace("\t", " ")).strip()


def load_json(path):
    with open(path, "r", encoding="utf-8") as file:
        return json.load(file)


def require_columns(frame, required, frame_name):
    missing = [column for column in required if column not in frame.columns]
    if missing:
        raise RuntimeError(f"{frame_name} missing required columns: {missing}")


def boolean_series(values):
    if pd.api.types.is_bool_dtype(values):
        return values.fillna(False).astype(bool)
    if pd.api.types.is_numeric_dtype(values):
        return values.fillna(0).astype(float).ne(0)
    normalized = values.fillna("").astype(str).str.strip().str.lower()
    return normalized.isin({"1", "true", "t", "yes", "y"})


def canonical_tokens(value):
    text = normalize_space(value).lower().replace("_", " ").replace("-", " ").replace("/", " ")
    return [TOKEN_EQUIVALENCE.get(token, token) for token in re.findall(r"[a-z0-9']+", text)]


def tokenize_sparse_document(value):
    return [token for token in canonical_tokens(value) if token not in LINGUISTIC_STOPWORDS]


def tokenize_sparse_query(value):
    stopwords = LINGUISTIC_STOPWORDS | GENERIC_ANCHOR_TOKENS
    return [token for token in canonical_tokens(value) if token not in stopwords]


def stable_unique(values):
    return list(dict.fromkeys(values))


def bm25_top_exact(bm25, query_tokens, top_k):
    scores = np.asarray(bm25.get_scores(query_tokens), dtype=np.float64)
    if not np.isfinite(scores).all():
        raise RuntimeError("BM25 scores contain non-finite values.")
    stable_position = np.arange(len(scores), dtype=np.int64)
    order = np.lexsort((stable_position, -scores))
    selected = order[:int(top_k)].astype(np.int64)
    return selected, scores[selected]


def top_exact_item_ids(bm25, query_tokens, item_ids, top_k):
    selected, _ = bm25_top_exact(bm25, query_tokens, top_k)
    return [item_ids[int(index)] for index in selected]


def rrf_fuse(source_lists, weights, top_k):
    if len(source_lists) != len(weights):
        raise RuntimeError("RRF source-list and weight counts differ.")
    scores = defaultdict(float)
    for source_items, weight in zip(source_lists, weights):
        for rank, item_id in enumerate(source_items, start=1):
            scores[item_id] += float(weight) / (RRF_K + rank)
    ranked = sorted(scores, key=lambda item_id: (-scores[item_id], item_id))[:top_k]
    return ranked, [float(scores[item_id]) for item_id in ranked]


def weighted_overlap(query_terms_value, item_terms, token_idf):
    query_term_set = set(query_terms_value)
    item_term_set = set(item_terms)
    if not query_term_set:
        return 0.0
    denominator = sum(token_idf.get(token, 1.0) for token in query_term_set)
    numerator = sum(token_idf.get(token, 1.0) for token in query_term_set & item_term_set)
    return float(numerator / max(denominator, 1e-12))


def softmax_weights(scores):
    values = np.asarray(scores, dtype=float)
    if len(values) == 0:
        return np.array([], dtype=float)
    shifted = values / ATTENTION_TEMPERATURE
    shifted -= shifted.max()
    weights = np.exp(shifted)
    return weights / weights.sum()


def normalized_entropy(weights):
    values = np.asarray(weights, dtype=float)
    values = values[values > 0]
    if len(values) <= 1:
        return 0.0
    return float(-np.sum(values * np.log(values)) / math.log(len(values)))


def contains_token_sequence(sequence, subsequence):
    if not subsequence or len(subsequence) > len(sequence):
        return False
    width = len(subsequence)
    return any(tuple(sequence[start:start + width]) == tuple(subsequence) for start in range(len(sequence) - width + 1))


def rank_metrics(rank, k=PRIMARY_K):
    hit = int(0 < int(rank) <= int(k))
    return {
        "hit_at_5": hit,
        "ndcg_at_5": float(1.0 / math.log2(int(rank) + 1)) if hit else 0.0,
        "mrr_at_5": float(1.0 / int(rank)) if hit else 0.0,
    }


def deterministic_seed(label):
    digest = hashlib.sha256(label.encode("utf-8")).hexdigest()
    return RANDOM_SEED + int(digest[:8], 16) % 1_000_000


def cluster_bootstrap_mean_ci(frame, value_column, cluster_column, label):
    work = frame[[cluster_column, value_column]].dropna().copy()
    grouped = [group[value_column].to_numpy(dtype=float) for _, group in work.groupby(cluster_column, sort=True)]
    if not grouped:
        return (np.nan, np.nan)
    if len(grouped) == 1:
        value = float(np.mean(grouped[0]))
        return (value, value)
    rng = np.random.default_rng(deterministic_seed(label))
    estimates = np.empty(BOOTSTRAP_REPS, dtype=float)
    for index in range(BOOTSTRAP_REPS):
        sampled = rng.integers(0, len(grouped), size=len(grouped))
        estimates[index] = float(np.mean(np.concatenate([grouped[position] for position in sampled])))
    alpha = 1.0 - BOOTSTRAP_CONFIDENCE
    return tuple(np.quantile(estimates, [alpha / 2.0, 1.0 - alpha / 2.0]).astype(float))


In [7]:
# =========================================================
# Load authoritative inputs and validate fixed contracts
# =========================================================
required_paths = [
    QUERY_CACHE_PATH, QUERY_CONTRACT_PATH, PRIOR_HISTORY_PATH, SAMPLING_MANIFEST_PATH,
    ITEM_DOCS_PATH, ITEM_FACETS_PATH, RETRIEVAL_ARTIFACT_MANIFEST_PATH,
    STAGE1_MANIFEST_PATH, STAGE1_WINNER_MANIFEST_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f"Missing required inputs: {missing_paths}")

query_contract = load_json(QUERY_CONTRACT_PATH)
sampling_manifest = load_json(SAMPLING_MANIFEST_PATH)
retrieval_manifest = load_json(RETRIEVAL_ARTIFACT_MANIFEST_PATH)
stage1_manifest = load_json(STAGE1_MANIFEST_PATH)
stage1_winner = load_json(STAGE1_WINNER_MANIFEST_PATH)

if query_contract.get("active_query_column") != ACTIVE_QUERY_COLUMN:
    raise RuntimeError("Notebook 06 active query column must be query.")
if query_contract.get("evidence_scope") != "target_review_safe_signals_only":
    raise RuntimeError("Notebook 06 query evidence scope mismatch.")
if query_contract.get("user_prior_evidence_used") is not False:
    raise RuntimeError("Notebook 06 synthetic query must not use user-prior evidence.")
if query_contract.get("historical_review_evidence_used") is not False:
    raise RuntimeError("Notebook 06 synthetic query must not use historical-review evidence.")

history_policy = normalize_space(
    sampling_manifest.get("training_prior_history_policy", sampling_manifest.get("training_prior_history_rule", ""))
).lower()
if not history_policy or "prior" not in history_policy or "target" not in history_policy:
    raise RuntimeError("Notebook 03 training-prior history policy is missing or ambiguous.")

required_retrieval_contract = {
    "evidence_scope": ITEM_EVIDENCE_SCOPE,
    "dense_source": DENSE_TEXT_COLUMN,
    "sparse_source": SPARSE_TEXT_COLUMN,
    "historical_review_reputation_enabled": True,
    "review_reputation_graph_enabled": True,
    "brand_in_functional_graph": False,
    "brand_graph_enabled": True,
    "brand_in_retrieval_text": True,
    "brand_in_profile_source_text": True,
    "brand_in_synthetic_query": False,
}
for key, expected in required_retrieval_contract.items():
    if retrieval_manifest.get(key) != expected:
        raise RuntimeError(f"Notebook 04 retrieval contract mismatch for {key}: {retrieval_manifest.get(key)!r}")

if stage1_winner.get("contract_version") != "stage1_query_only_winner_v1":
    raise RuntimeError("Notebook 07 winner contract version mismatch.")
if stage1_winner.get("category_id") != CATEGORY_ID:
    raise RuntimeError("Notebook 07 winner category mismatch.")
if normalize_space(stage1_winner.get("winner_method_key")) != EXPECTED_BASELINE_METHOD_KEY:
    raise RuntimeError("The fixed category-specific query-only winner does not match the thesis contract.")
if normalize_space(stage1_winner.get("winner_method_label")) != EXPECTED_BASELINE_METHOD_LABEL:
    raise RuntimeError("The fixed category-specific query-only winner label does not match the thesis contract.")
if stage1_winner.get("user_prior_enabled") is not False:
    raise RuntimeError("No Prior must remain query-only.")
if stage1_winner.get("candidate_budget_policy") != "exact_k_all_methods":
    raise RuntimeError("Notebook 07 winner must use exact-K candidate budgets.")
if int(stage1_winner.get("effective_candidate_count_per_query", -1)) < POOL_DEPTH:
    raise RuntimeError("Notebook 07 winner cache is shallower than the fixed diagnostic depth 1000.")
if int(stage1_manifest.get("candidate_budget_k", -1)) < POOL_DEPTH:
    raise RuntimeError("Notebook 07 run depth is shallower than the fixed diagnostic depth 1000.")
if stage1_manifest.get("user_prior_enabled") is not False:
    raise RuntimeError("Notebook 07 baseline must not use user prior.")

baseline_path_text = normalize_space(stage1_winner.get("winner_candidate_path"))
if not baseline_path_text:
    raise RuntimeError("Notebook 07 winner candidate path is empty.")
BASELINE_CANDIDATES_PATH = Path(baseline_path_text)
if not BASELINE_CANDIDATES_PATH.exists():
    raise FileNotFoundError(f"Notebook 07 winner candidate cache is missing: {BASELINE_CANDIDATES_PATH}")

query_schema = pq.ParquetFile(QUERY_CACHE_PATH).schema.names
required_query_columns = [
    "case_id", "user_id", "target_parent_asin", "regime", "target_timestamp_ms", ACTIVE_QUERY_COLUMN,
    "query_evidence_source", "target_metadata_fallback_used", "item_context_fallback_used",
    "historical_review_evidence_used", "user_prior_evidence_used", "insufficient_review_evidence",
    "query_clean_is_active", *QUERY_PASSTHROUGH_COLUMNS,
]
if COMPATIBILITY_QUERY_ALIAS is not None:
    required_query_columns.append(COMPATIBILITY_QUERY_ALIAS)
if "query_id" in query_schema:
    required_query_columns.append("query_id")
queries = pd.read_parquet(QUERY_CACHE_PATH, columns=required_query_columns).copy()
require_columns(queries, required_query_columns, "query cache")
for column in ["case_id", "user_id", "target_parent_asin", "regime", ACTIVE_QUERY_COLUMN]:
    queries[column] = queries[column].fillna("").astype(str).map(normalize_space)
queries["target_timestamp_ms"] = pd.to_numeric(queries["target_timestamp_ms"], errors="raise").astype("int64")
queries["active_query_text"] = queries[ACTIVE_QUERY_COLUMN]
if "query_id" in queries.columns:
    queries["query_id"] = queries["query_id"].fillna("").astype(str).map(normalize_space)
    QUERY_ID_SOURCE = "query_cache"
else:
    queries["query_id"] = queries["case_id"]
    QUERY_ID_SOURCE = "case_id_alias"
if queries["case_id"].duplicated().any():
    raise RuntimeError("Query grain must be one row per case_id.")
if queries["query_id"].eq("").any() or queries["query_id"].duplicated().any():
    raise RuntimeError("query_id must be non-empty and one-to-one with case_id.")
if queries[["case_id", "query_id"]].drop_duplicates().shape[0] != len(queries):
    raise RuntimeError("case_id and query_id do not form a one-to-one mapping.")
if queries[["case_id", "user_id", "target_parent_asin", "regime", "active_query_text"]].eq("").any().any():
    raise RuntimeError("Query cache contains an empty required identifier or query.")
if not set(queries["regime"]).issubset(set(REGIME_ORDER)):
    raise RuntimeError(f"Unexpected regimes: {sorted(set(queries['regime']) - set(REGIME_ORDER))}")
if not queries["query_evidence_source"].eq("target_review_safe_signals_only").all():
    raise RuntimeError("Every query must use target-review-safe signals only.")
for column in [
    "target_metadata_fallback_used", "item_context_fallback_used", "historical_review_evidence_used",
    "user_prior_evidence_used", "insufficient_review_evidence", "query_clean_is_active",
]:
    if boolean_series(queries[column]).any():
        raise RuntimeError(f"Query audit field must be false: {column}")
if COMPATIBILITY_QUERY_ALIAS is not None:
    alias = queries[COMPATIBILITY_QUERY_ALIAS].fillna("").astype(str).map(normalize_space)
    if not alias.eq(queries[ACTIVE_QUERY_COLUMN]).all():
        raise RuntimeError(f"{COMPATIBILIT_QUER_ALIAS} must equal query.")

item_docs = pd.read_parquet(
    ITEM_DOCS_PATH,
    columns=["parent_asin", DENSE_TEXT_COLUMN, SPARSE_TEXT_COLUMN, BRAND_TEXT_COLUMN, PROFILE_SAFE_TEXT_COLUMN],
).copy()
for column in ["parent_asin", DENSE_TEXT_COLUMN, SPARSE_TEXT_COLUMN, BRAND_TEXT_COLUMN, PROFILE_SAFE_TEXT_COLUMN]:
    item_docs[column] = item_docs[column].fillna("").astype(str).map(normalize_space)
if item_docs["parent_asin"].eq("").any() or item_docs["parent_asin"].duplicated().any():
    raise RuntimeError("Item documents contain empty or duplicate parent_asin values.")
if item_docs[DENSE_TEXT_COLUMN].eq("").any() or item_docs[SPARSE_TEXT_COLUMN].eq("").any():
    raise RuntimeError("Every item must retain non-empty dense and sparse retrieval text.")
if item_docs[BRAND_TEXT_COLUMN].eq("").all():
    raise RuntimeError("Notebook 04 item representation must retain brand evidence.")
item_docs = item_docs.sort_values("parent_asin", kind="stable").reset_index(drop=True)
if len(item_docs) < POOL_DEPTH:
    raise RuntimeError("Catalog is smaller than the fixed candidate budget 700.")
item_ids = item_docs["parent_asin"].tolist()
item_id_set = set(item_ids)
item_brand_map = item_docs.set_index("parent_asin")[BRAND_TEXT_COLUMN].to_dict()

facet_columns = [
    "parent_asin", "facet_value_norm", "facet_role", "is_brand", "is_review_derived",
    "is_product_functional_facet", "is_query_safe", "is_generic_category_anchor",
    "is_generic_utility_token", "is_context_dependent_utility_token", "is_metadata_facet_source",
    "is_disallowed_nonfacet_source", "is_core_graph_facet", "is_brand_graph_facet",
    "is_retrieval_safe", "is_profile_safe",
]
facets = pd.read_parquet(ITEM_FACETS_PATH, columns=facet_columns).copy()
require_columns(facets, facet_columns, "item facets")

prior_schema = pq.ParquetFile(PRIOR_HISTORY_PATH).schema.names
forbidden_prior_fragments = (
    "review_text", "review_body", "review_title", "raw_review", "brand", "manufacturer", "seller",
    "rating", "sentiment", "prompt", "response", "llm",
)
forbidden_prior_columns = sorted(
    column for column in prior_schema if any(fragment in column.lower() for fragment in forbidden_prior_fragments)
)
if forbidden_prior_columns:
    raise RuntimeError(f"Prior-history artifact contains forbidden evidence fields: {forbidden_prior_columns}")
prior_columns = ["case_id", "user_id", "target_timestamp_ms", "prior_item_id", "prior_timestamp_ms"]
if PRIOR_HAS_TARGET_PARENT_ASIN:
    prior_columns.append("target_parent_asin")
prior_history = pd.read_parquet(PRIOR_HISTORY_PATH, columns=prior_columns).copy()
require_columns(prior_history, prior_columns, "prior history")

baseline_columns = [
    "case_id", "user_id", "regime", "target_parent_asin", "candidate_parent_asin", "candidate_rank",
    "candidate_score", "method_key", "retrieval_method", "score_semantics", "candidate_brand_facet_text", "is_target",
]
baseline_schema = pq.ParquetFile(BASELINE_CANDIDATES_PATH).schema.names
if "query_id" in baseline_schema:
    baseline_columns.append("query_id")
missing_baseline = [column for column in baseline_columns if column not in baseline_schema]
if missing_baseline:
    raise RuntimeError(f"Notebook 07 winner candidates are missing columns: {missing_baseline}")
baseline_candidates = pd.read_parquet(BASELINE_CANDIDATES_PATH, columns=baseline_columns).copy()
for column in [
    "case_id", "user_id", "regime", "target_parent_asin", "candidate_parent_asin", "method_key",
    "retrieval_method", "score_semantics", "candidate_brand_facet_text",
]:
    baseline_candidates[column] = baseline_candidates[column].fillna("").astype(str).map(normalize_space)
if "query_id" in baseline_candidates.columns:
    baseline_candidates["query_id"] = baseline_candidates["query_id"].fillna("").astype(str).map(normalize_space)
else:
    baseline_candidates["query_id"] = baseline_candidates["case_id"]
baseline_candidates["candidate_rank"] = pd.to_numeric(baseline_candidates["candidate_rank"], errors="raise").astype(int)
baseline_candidates["candidate_score"] = pd.to_numeric(baseline_candidates["candidate_score"], errors="raise").astype(float)
baseline_candidates["is_target"] = boolean_series(baseline_candidates["is_target"])
baseline_candidates = baseline_candidates.loc[baseline_candidates["candidate_rank"].le(POOL_DEPTH)].copy()
baseline_candidates = baseline_candidates.sort_values(["case_id", "candidate_rank"], kind="stable").reset_index(drop=True)

query_by_case = queries.set_index("case_id")
if set(baseline_candidates["case_id"]) != set(queries["case_id"]):
    raise RuntimeError("Notebook 07 baseline and query-cache case universes differ.")
for column, query_column in [("query_id", "query_id"), ("user_id", "user_id"), ("regime", "regime"), ("target_parent_asin", "target_parent_asin")]:
    expected = baseline_candidates["case_id"].map(query_by_case[query_column])
    if not baseline_candidates[column].eq(expected).all():
        raise RuntimeError(f"Notebook 07 baseline {column} does not match the case-grain query metadata.")
if not baseline_candidates["method_key"].eq(EXPECTED_BASELINE_METHOD_KEY).all():
    raise RuntimeError("Baseline candidate cache method key differs from the fixed No Prior winner.")
if not baseline_candidates["retrieval_method"].eq(EXPECTED_BASELINE_METHOD_LABEL).all():
    raise RuntimeError("Baseline candidate cache method label differs from the fixed No Prior winner.")
expected_brand = baseline_candidates["candidate_parent_asin"].map(item_brand_map).fillna("")
if not baseline_candidates["candidate_parent_asin"].isin(item_id_set).all():
    raise RuntimeError("Baseline candidate cache contains an item outside the Notebook 04 retrieval catalog.")
if not queries["target_parent_asin"].isin(item_id_set).all():
    raise RuntimeError("A held-out target is outside the Notebook 04 retrieval catalog.")
if not baseline_candidates["candidate_brand_facet_text"].eq(expected_brand).all():
    raise RuntimeError("Baseline candidate brands differ from Notebook 04 item representation.")
expected_is_target = baseline_candidates["candidate_parent_asin"].eq(baseline_candidates["target_parent_asin"])
if not baseline_candidates["is_target"].eq(expected_is_target).all():
    raise RuntimeError("Baseline is_target flags do not match the held-out target identifier.")
if baseline_candidates.duplicated(["case_id", "candidate_parent_asin"]).any():
    raise RuntimeError("Baseline contains within-case duplicate candidates.")
for case_id, group in baseline_candidates.groupby("case_id", sort=False):
    if group["candidate_rank"].tolist() != list(range(1, POOL_DEPTH + 1)):
        raise RuntimeError(f"Baseline ranks are not exactly 1..700 for case {case_id}.")
    scores = group["candidate_score"].to_numpy(dtype=float)
    if not np.isfinite(scores).all() or np.any(np.diff(scores) > 1e-12):
        raise RuntimeError(f"Baseline scores are invalid for case {case_id}.")

print("Cases:", len(queries))
print("Catalog items:", len(item_docs))
print("Fixed No Prior winner:", EXPECTED_BASELINE_METHOD_KEY, "-", EXPECTED_BASELINE_METHOD_LABEL)
print("Input contract validation: passed")


Cases: 2288
Catalog items: 77502
Fixed No Prior winner: graph_hybrid - Graph-Hybrid
Input contract validation: passed


In [8]:
# =========================================================
# Strict history, leakage checks, and profile-safe facet index
# =========================================================
query_case_ids = set(queries["case_id"])
query_users = queries.set_index("case_id")["user_id"]
query_targets = queries.set_index("case_id")["target_parent_asin"]
query_timestamps = queries.set_index("case_id")["target_timestamp_ms"]

prior_id_columns = ["case_id", "user_id", "prior_item_id"]
if PRIOR_HAS_TARGET_PARENT_ASIN:
    prior_id_columns.append("target_parent_asin")
for column in prior_id_columns:
    prior_history[column] = prior_history[column].fillna("").astype(str).map(normalize_space)
for column in ["target_timestamp_ms", "prior_timestamp_ms"]:
    prior_history[column] = pd.to_numeric(prior_history[column], errors="raise").astype("int64")
prior_history = prior_history.loc[prior_history["case_id"].isin(query_case_ids) & prior_history["prior_item_id"].ne("")].copy()

strict_prior_event_key = ["case_id", "user_id", "prior_item_id", "prior_timestamp_ms"]
prior_duplicate_mask = prior_history.duplicated(strict_prior_event_key, keep=False)
prior_duplicate_row_count = int(prior_duplicate_mask.sum())
prior_duplicate_event_count = (
    int(prior_history.loc[prior_duplicate_mask, strict_prior_event_key].drop_duplicates().shape[0])
    if prior_duplicate_row_count
    else 0
)

if prior_duplicate_row_count:
    prior_duplicate_qc = (
        prior_history.loc[prior_duplicate_mask, strict_prior_event_key + ["target_timestamp_ms"]]
        .groupby(strict_prior_event_key + ["target_timestamp_ms"], dropna=False)
        .size()
        .reset_index(name="artifact_row_count")
        .sort_values(["artifact_row_count", "case_id", "prior_timestamp_ms", "prior_item_id"], ascending=[False, True,
        True, True])
        .reset_index(drop=True)
    )
    display(prior_duplicate_qc.head(20))

if len(prior_history):
    if not prior_history["user_id"].eq(prior_history["case_id"].map(query_users)).all():
        raise RuntimeError("Prior-history user_id does not match query case grain.")
    if PRIOR_HAS_TARGET_PARENT_ASIN:
        if not prior_history["target_parent_asin"].eq(prior_history["case_id"].map(query_targets)).all():
            raise RuntimeError("Prior-history target item does not match query case grain.")
    if not prior_history["target_timestamp_ms"].eq(prior_history["case_id"].map(query_timestamps)).all():
        raise RuntimeError("Prior-history target timestamp does not match query case grain.")
    if not prior_history["prior_timestamp_ms"].lt(prior_history["target_timestamp_ms"]).all():
        raise RuntimeError("Target/future leakage: a prior interaction is not strictly before its target.")
    if prior_history["prior_item_id"].eq(prior_history["case_id"].map(query_targets)).any():
        raise RuntimeError("Target leakage: prior history contains the held-out target item.")

raw_prior_counts = prior_history.groupby("case_id").size()
history_exclusions = prior_history.loc[~prior_history["prior_item_id"].isin(item_id_set)].copy()
history_exclusions["exclusion_reason"] = "prior_item_outside_notebook04_retrieval_catalog"
eligible_history = prior_history.loc[prior_history["prior_item_id"].isin(item_id_set)].copy()
eligible_history = eligible_history.sort_values(
    ["case_id", "prior_timestamp_ms", "prior_item_id"], ascending=[True, False, True], kind="stable"
).reset_index(drop=True)

# QCHS preserves the canonical Notebook 08 unique-item selection unit.
prior_items = (
    eligible_history.groupby(["case_id", "user_id", "prior_item_id"], as_index=False)
    .agg(
        prior_review_count=("prior_timestamp_ms", "size"),
        latest_prior_timestamp_ms=("prior_timestamp_ms", "max"),
        target_timestamp_ms=("target_timestamp_ms", "first"),
    )
    .sort_values(["case_id", "latest_prior_timestamp_ms", "prior_item_id"], ascending=[True, False, True], kind="stable")
    .reset_index(drop=True)
)

# All Prior deliberately retains every eligible strict interaction; item-level deduplication is disabled.
all_prior_events_by_case = {
    case_id: group[["prior_item_id", "prior_timestamp_ms"]].to_dict("records")
    for case_id, group in eligible_history.groupby("case_id", sort=False)
}
qchs_items_by_case = {
    case_id: group[["prior_item_id", "latest_prior_timestamp_ms", "prior_review_count"]].to_dict("records")
    for case_id, group in prior_items.groupby("case_id", sort=False)
}

queries["raw_prior_event_count"] = queries["case_id"].map(raw_prior_counts).fillna(0).astype(int)
queries["eligible_prior_event_count"] = queries["case_id"].map(eligible_history.groupby("case_id").size()).fillna(0).astype(int)
queries["eligible_prior_unique_item_count"] = queries["case_id"].map(prior_items.groupby("case_id").size()).fillna(0).astype(int)
queries["outside_catalog_prior_event_count"] = queries["raw_prior_event_count"] - queries["eligible_prior_event_count"]

if queries.loc[queries["regime"].eq("cold"), "raw_prior_event_count"].ne(0).any():
    raise RuntimeError("Cold cases must have zero strict pre-target history.")
if queries.loc[queries["regime"].ne("cold"), "raw_prior_event_count"].lt(1).any():
    raise RuntimeError("Every non-cold case must have at least one strict pre-target interaction.")

for column in [
    "is_brand", "is_review_derived", "is_product_functional_facet", "is_query_safe",
    "is_generic_category_anchor", "is_generic_utility_token", "is_context_dependent_utility_token",
    "is_metadata_facet_source", "is_disallowed_nonfacet_source", "is_core_graph_facet",
    "is_brand_graph_facet", "is_retrieval_safe", "is_profile_safe",
]:
    facets[column] = boolean_series(facets[column])
facets["parent_asin"] = facets["parent_asin"].fillna("").astype(str).map(normalize_space)
facets["facet_role"] = facets["facet_role"].fillna("").astype(str).map(normalize_space).str.lower()
facets["facet_value_norm"] = facets["facet_value_norm"].fillna("").astype(str).map(normalize_space).str.lower()

explicit_metadata_mask = (
    facets["is_product_functional_facet"] & facets["is_query_safe"] & ~facets["is_brand"]
    & ~facets["is_review_derived"] & ~facets["is_generic_category_anchor"]
    & ~facets["is_generic_utility_token"] & ~facets["is_context_dependent_utility_token"]
    & facets["is_metadata_facet_source"] & ~facets["is_disallowed_nonfacet_source"]
)
explicit_brand_mask = (
    facets["facet_role"].eq("brand") & facets["is_brand"] & ~facets["is_review_derived"]
    & ~facets["is_generic_category_anchor"] & ~facets["is_generic_utility_token"]
    & ~facets["is_context_dependent_utility_token"] & facets["is_metadata_facet_source"]
    & ~facets["is_disallowed_nonfacet_source"]
)
explicit_profile_safe_mask = explicit_metadata_mask | explicit_brand_mask
if not facets["is_core_graph_facet"].eq(explicit_metadata_mask).all():
    raise RuntimeError("Notebook 04 functional-facet mask contract changed.")
if not facets["is_brand_graph_facet"].eq(explicit_brand_mask).all():
    raise RuntimeError("Notebook 04 brand-facet mask contract changed.")
if not facets["is_profile_safe"].eq(explicit_profile_safe_mask).all():
    raise RuntimeError("Notebook 04 profile-safe facet contract changed.")

alignment_facets = facets.loc[
    explicit_metadata_mask & facets["parent_asin"].isin(item_id_set) & facets["facet_value_norm"].ne("")
].copy()
profile_safe_facets = facets.loc[
    explicit_profile_safe_mask & facets["parent_asin"].isin(item_id_set) & facets["facet_value_norm"].ne("")
].copy()
if alignment_facets.empty or profile_safe_facets.empty:
    raise RuntimeError("Required alignment or profile-safe facets are empty.")
if alignment_facets["is_brand"].any() or alignment_facets["is_review_derived"].any():
    raise RuntimeError("Brand or review-derived evidence entered QCHS alignment.")
if profile_safe_facets["is_review_derived"].any():
    raise RuntimeError("Historical population-review facets entered the user-profile channel.")
if not alignment_facets["facet_role"].isin(FUNCTIONAL_ROLES).all():
    raise RuntimeError("Unexpected functional role in QCHS alignment.")
if not profile_safe_facets["facet_role"].isin(PROFILE_ROLES).all():
    raise RuntimeError("Unexpected role in profile-safe evidence.")
brand_facets = profile_safe_facets.loc[profile_safe_facets["is_brand"]]
if brand_facets.empty or brand_facets["is_query_safe"].any() or brand_facets["is_product_functional_facet"].any():
    raise RuntimeError("Brand must be profile-safe, query-unsafe, and distinct from functional facets.")

alignment_facets["facet_key"] = alignment_facets["facet_role"] + "::" + alignment_facets["facet_value_norm"]
profile_safe_facets["facet_key"] = profile_safe_facets["facet_role"] + "::" + profile_safe_facets["facet_value_norm"]
alignment_facets = alignment_facets.drop_duplicates(["parent_asin", "facet_key"]).copy()
profile_safe_facets = profile_safe_facets.drop_duplicates(["parent_asin", "facet_key"]).copy()

profile_df = profile_safe_facets.groupby("facet_key")["parent_asin"].nunique()
profile_facet_idf = {key: float(math.log((1.0 + len(item_docs)) / (1.0 + count)) + 1.0) for key, count in profile_df.items()}
alignment_df = alignment_facets.groupby("facet_key")["parent_asin"].nunique()
alignment_facet_idf = {key: float(math.log((1.0 + len(item_docs)) / (1.0 + count)) + 1.0) for key, count in alignment_df.items()}

alignment_item_token_sets = {}
alignment_records_by_item = defaultdict(list)
alignment_token_document_frequency = Counter()
for item_id, group in alignment_facets.groupby("parent_asin", sort=False):
    item_tokens = set()
    for row in group.itertuples(index=False):
        full_phrase_tokens = tuple(canonical_tokens(row.facet_value_norm))
        phrase_tokens = tuple(token for token in full_phrase_tokens if token not in ATTENTION_STOPWORDS)
        if not full_phrase_tokens or not phrase_tokens:
            continue
        item_tokens.update(phrase_tokens)
        alignment_records_by_item[item_id].append({
            "facet_key": row.facet_key, "facet_role": row.facet_role, "phrase": row.facet_value_norm,
            "phrase_tokens": full_phrase_tokens, "tokens": phrase_tokens,
            "idf": alignment_facet_idf[row.facet_key], "is_brand": False,
        })
    alignment_item_token_sets[item_id] = item_tokens
    alignment_token_document_frequency.update(item_tokens)
alignment_token_idf = {
    token: float(math.log((1.0 + len(item_docs)) / (1.0 + count)) + 1.0)
    for token, count in alignment_token_document_frequency.items()
}

facet_records_by_item = defaultdict(list)
for item_id, group in profile_safe_facets.groupby("parent_asin", sort=False):
    for row in group.itertuples(index=False):
        full_phrase_tokens = tuple(canonical_tokens(row.facet_value_norm))
        phrase_tokens = tuple(token for token in full_phrase_tokens if token not in ATTENTION_STOPWORDS)
        if not full_phrase_tokens:
            continue
        facet_records_by_item[item_id].append({
            "facet_key": row.facet_key, "facet_role": row.facet_role, "phrase": row.facet_value_norm,
            "phrase_tokens": full_phrase_tokens, "tokens": phrase_tokens,
            "idf": profile_facet_idf[row.facet_key], "is_brand": bool(row.is_brand),
        })

print("Strict prior events:", len(prior_history))
print("Eligible in-catalog prior events:", len(eligible_history))
print("Excluded outside-catalog events:", len(history_exclusions))
print("Leakage and profile-safe evidence validation: passed")


,case_id,user_id,prior_item_id,prior_timestamp_ms,target_timestamp_ms,artifact_row_count
0,59f1f5660f5d,AHPSEHAIFSPE2DIVR4BOEESBRUHQ,B07XFLYN6X,1607885293539,1679503367809,2
1,59f1f5660f5d,AHPSEHAIFSPE2DIVR4BOEESBRUHQ,B01N13W31F,1607889354294,1679503367809,2
2,61418f7d08d7,AHMMZDH5QMVWL2MGHZSKQCNMNXKA,B07B4KQVK6,1666491978220,1680192657607,2
3,61418f7d08d7,AHMMZDH5QMVWL2MGHZSKQCNMNXKA,B07XNCF447,1668491273806,1680192657607,2
4,61418f7d08d7,AHMMZDH5QMVWL2MGHZSKQCNMNXKA,B0C6L7J66K,1668491601660,1680192657607,2
5,6e033b9e4816,AFROHJNZG5WDHZQB2UU7KWPTBBIQ,B01HWE4XIQ,1628362124769,1674606722576,2
6,6e033b9e4816,AFROHJNZG5WDHZQB2UU7KWPTBBIQ,B0C6XSN22X,1663632869965,1674606722576,2
7,9b554e9b0dfe,AE2Z623TMMSSRFZ4DXBFB4AFDR2Q,B0BY2ZYZZ8,1577564825118,1677963059229,2
8,9b554e9b0dfe,AE2Z623TMMSSRFZ4DXBFB4AFDR2Q,B01BE73K8Q,1591398807416,1677963059229,2
9,9b554e9b0dfe,AE2Z623TMMSSRFZ4DXBFB4AFDR2Q,B09CFW68KP,1665878737038,1677963059229,2


Strict prior events: 22540
Eligible in-catalog prior events: 22519
Excluded outside-catalog events: 21
Leakage and profile-safe evidence validation: passed


In [9]:
# =========================================================
# Fixed QCHS and All Prior profile policies
# =========================================================
def prior_alignment(query_text, item_id):
    query_token_sequence = canonical_tokens(query_text)
    terms = stable_unique(
        token for token in query_token_sequence if token not in ATTENTION_STOPWORDS and len(token) > 1
    )
    records = alignment_records_by_item.get(item_id, [])
    if not terms or not records:
        return 0.0
    item_tokens = alignment_item_token_sets.get(item_id, set())
    token_score = weighted_overlap(terms, item_tokens, alignment_token_idf)
    exact_phrase_score = float(
        any(contains_token_sequence(query_token_sequence, record["phrase_tokens"]) for record in records)
    )
    matched_roles = {
        record["facet_role"] for record in records if set(record["tokens"]) & set(terms)
    }
    role_score = min(len(matched_roles) / 2.0, 1.0)
    return float(0.70 * token_score + 0.20 * exact_phrase_score + 0.10 * role_score)


def selected_facet_keys(selected_items):
    return stable_unique(
        record["facet_key"] for item_id in selected_items for record in facet_records_by_item.get(item_id, [])
    )


def build_anchors(selected_items, item_weights):
    anchor_scores = defaultdict(float)
    anchor_support = defaultdict(set)
    anchor_meta = {}
    for item_id, item_weight in zip(selected_items, item_weights):
        for record in facet_records_by_item.get(item_id, []):
            key = record["facet_key"]
            anchor_scores[key] += float(item_weight) * float(record["idf"])
            anchor_support[key].add(item_id)
            anchor_meta[key] = (record["facet_role"], record["phrase"], bool(record["is_brand"]))
    ranked_keys = sorted(
        anchor_scores,
        key=lambda key: (
            -(anchor_scores[key] + 0.05 * math.log1p(len(anchor_support[key]))),
            anchor_meta[key][0], anchor_meta[key][1],
        ),
    )
    selected_keys, selected_phrases, selected_roles = [], [], []
    role_counts = Counter()
    for key in ranked_keys:
        role, phrase, _ = anchor_meta[key]
        if role_counts[role] >= MAX_ANCHOR_PHRASES_PER_ROLE:
            continue
        selected_keys.append(key)
        selected_phrases.append(phrase)
        selected_roles.append(role)
        role_counts[role] += 1
        if len(selected_keys) >= MAX_ANCHOR_PHRASES:
            break
    return selected_keys, selected_phrases, selected_roles


def empty_profile():
    return {
        "selected_items": [], "alignment_scores": [], "item_weights": [], "selected_facet_keys": [],
        "anchor_keys": [], "anchor_phrases": [], "anchor_roles": [], "attention_entropy": 0.0,
    }


def build_qchs_profile(query_text, history_items):
    scored = []
    for row in history_items:
        score = prior_alignment(query_text, row["prior_item_id"])
        if score > 0:
            scored.append((score, int(row["latest_prior_timestamp_ms"]), row["prior_item_id"]))
    scored.sort(key=lambda value: (-value[0], -value[1], value[2]))
    scored = scored[:MAX_QCHS_PRIOR_ITEMS]
    item_ids = [item_id for _, _, item_id in scored]
    scores = [score for score, _, _ in scored]
    if not item_ids:
        return empty_profile()
    weights = softmax_weights(scores)
    anchor_keys, anchor_phrases, anchor_roles = build_anchors(item_ids, weights)
    return {
        "selected_items": item_ids, "alignment_scores": scores, "item_weights": weights.tolist(),
        "selected_facet_keys": selected_facet_keys(item_ids), "anchor_keys": anchor_keys,
        "anchor_phrases": anchor_phrases, "anchor_roles": anchor_roles,
        "attention_entropy": normalized_entropy(weights),
    }


def build_all_prior_profile(history_events):
    # Repeated prior-item interactions remain repeated: this is an event-level All Prior control.
    item_ids = [row["prior_item_id"] for row in history_events]
    if not item_ids:
        return empty_profile()
    weights = np.full(len(item_ids), 1.0 / len(item_ids), dtype=float)
    anchor_keys, anchor_phrases, anchor_roles = build_anchors(item_ids, weights)
    return {
        "selected_items": item_ids, "alignment_scores": [0.0] * len(item_ids),
        "item_weights": weights.tolist(), "selected_facet_keys": selected_facet_keys(item_ids),
        "anchor_keys": anchor_keys, "anchor_phrases": anchor_phrases, "anchor_roles": anchor_roles,
        "attention_entropy": normalized_entropy(weights),
    }


profiles = {"qchs": {}, "all_prior": {}}
profile_rows = []
event_count_by_case_item = eligible_history.groupby(["case_id", "prior_item_id"]).size().to_dict()

for row in queries.itertuples(index=False):
    qchs_profile = build_qchs_profile(row.active_query_text, qchs_items_by_case.get(row.case_id, []))
    all_prior_profile = build_all_prior_profile(all_prior_events_by_case.get(row.case_id, []))
    profiles["qchs"][row.case_id] = qchs_profile
    profiles["all_prior"][row.case_id] = all_prior_profile

    qchs_items = qchs_profile["selected_items"]
    all_prior_items = all_prior_profile["selected_items"]
    qchs_selected_event_count = int(sum(event_count_by_case_item.get((row.case_id, item_id), 0) for item_id in qchs_items))
    qchs_active = bool(qchs_items and qchs_profile["selected_facet_keys"] and qchs_profile["anchor_phrases"])
    all_prior_active = bool(all_prior_items and all_prior_profile["selected_facet_keys"] and all_prior_profile["anchor_phrases"])
    qchs_reason = "" if qchs_active else ("cold_no_history" if row.regime == "cold" else "no_qchs_aligned_profile_evidence")
    all_prior_reason = "" if all_prior_active else ("cold_no_history" if row.regime == "cold" else "no_profile_safe_prior_evidence")

    profile_rows.append({
        "case_id": row.case_id,
        "query_id": row.query_id,
        "user_id": row.user_id,
        "regime": row.regime,
        "target_parent_asin": row.target_parent_asin,
        "raw_prior_event_count": int(row.raw_prior_event_count),
        "eligible_prior_event_count": int(row.eligible_prior_event_count),
        "eligible_prior_unique_item_count": int(row.eligible_prior_unique_item_count),
        "outside_catalog_prior_event_count": int(row.outside_catalog_prior_event_count),
        "qchs_selected_unique_item_count": int(len(qchs_items)),
        "qchs_selected_event_coverage_count": qchs_selected_event_count,
        "qchs_unique_item_retention_rate": float(len(qchs_items) / row.eligible_prior_unique_item_count) if row.eligible_prior_unique_item_count else 0.0,
        "qchs_event_coverage_rate": float(qchs_selected_event_count / row.eligible_prior_event_count) if row.eligible_prior_event_count else 0.0,
        "qchs_alignment_mean": float(np.mean(qchs_profile["alignment_scores"])) if qchs_profile["alignment_scores"] else 0.0,
        "qchs_alignment_max": float(np.max(qchs_profile["alignment_scores"])) if qchs_profile["alignment_scores"] else 0.0,
        "qchs_attention_entropy": float(qchs_profile["attention_entropy"]),
        "qchs_selected_facet_count": int(len(qchs_profile["selected_facet_keys"])),
        "qchs_anchor_phrase_count": int(len(qchs_profile["anchor_phrases"])),
        "qchs_anchor_brand_count": int(sum(role == "brand" for role in qchs_profile["anchor_roles"])),
        "qchs_selected_prior_items": list(qchs_items),
        "qchs_anchor_phrases": list(qchs_profile["anchor_phrases"]),
        "qchs_active": qchs_active,
        "qchs_fallback_flag": not qchs_active,
        "qchs_fallback_reason": qchs_reason,
        "all_prior_selected_event_count": int(len(all_prior_items)),
        "all_prior_selected_unique_item_count": int(len(set(all_prior_items))),
        "all_prior_event_retention_rate": float(len(all_prior_items) / row.eligible_prior_event_count) if row.eligible_prior_event_count else 0.0,
        "all_prior_unique_item_retention_rate": float(len(set(all_prior_items)) / row.eligible_prior_unique_item_count) if row.eligible_prior_unique_item_count else 0.0,
        "all_prior_attention_entropy": float(all_prior_profile["attention_entropy"]),
        "all_prior_selected_facet_count": int(len(all_prior_profile["selected_facet_keys"])),
        "all_prior_anchor_phrase_count": int(len(all_prior_profile["anchor_phrases"])),
        "all_prior_anchor_brand_count": int(sum(role == "brand" for role in all_prior_profile["anchor_roles"])),
        "all_prior_selected_prior_event_items": list(all_prior_items),
        "all_prior_anchor_phrases": list(all_prior_profile["anchor_phrases"]),
        "all_prior_active": all_prior_active,
        "all_prior_fallback_flag": not all_prior_active,
        "all_prior_fallback_reason": all_prior_reason,
    })

profile_diagnostics = pd.DataFrame(profile_rows)
# Explicit stage-specific aliases prevent Stage-1 and Stage-2 QCHS populations from being conflated.
profile_diagnostics["stage1_qchs_active"] = profile_diagnostics["qchs_active"].astype(bool)
profile_diagnostics["stage1_qchs_fallback"] = ~profile_diagnostics["stage1_qchs_active"]
if profile_diagnostics.duplicated("case_id").any() or len(profile_diagnostics) != len(queries):
    raise RuntimeError("Profile diagnostics must remain one row per case_id.")
if not profile_diagnostics["all_prior_selected_event_count"].eq(profile_diagnostics["eligible_prior_event_count"]).all():
    raise RuntimeError("All Prior failed to retain every eligible strict pre-target interaction.")
if not profile_diagnostics["all_prior_selected_unique_item_count"].eq(profile_diagnostics["eligible_prior_unique_item_count"]).all():
    raise RuntimeError("All Prior failed to retain every eligible prior item.")
if profile_diagnostics.loc[profile_diagnostics["regime"].eq("cold"), ["qchs_active", "all_prior_active"]].any().any():
    raise RuntimeError("Cold cases must not activate either prior policy.")

print("Stage1-QCHS-active cases:", int(profile_diagnostics["qchs_active"].sum()))
print("All Prior-active cases:", int(profile_diagnostics["all_prior_active"].sum()))
print("All Prior event-retention contract: passed")


Stage1-QCHS-active cases: 1513
All Prior-active cases: 1716
All Prior event-retention contract: passed


In [ ]:
# =========================================================
# Matched retrieval sources and fixed RRF fusion at depth 1000
# =========================================================
baseline_items_by_case = baseline_candidates.groupby("case_id", sort=False)["candidate_parent_asin"].agg(list).to_dict()
baseline_scores_by_case = baseline_candidates.groupby("case_id", sort=False)["candidate_score"].agg(list).to_dict()
query_lookup = queries.set_index("case_id")
profile_lookup = profile_diagnostics.set_index("case_id")

sparse_corpus_tokens = [tokenize_sparse_document(text) for text in tqdm(item_docs[SPARSE_TEXT_COLUMN], desc="Sparse documents")]
if any(len(tokens) == 0 for tokens in sparse_corpus_tokens):
    raise RuntimeError("Every sparse item document must contain an indexed token.")
bm25 = BM25Okapi(sparse_corpus_tokens)

sparse_items = {policy: {} for policy in ["qchs", "all_prior"]}
expanded_dense_text = {policy: {} for policy in ["qchs", "all_prior"]}
for policy in ["qchs", "all_prior"]:
    for case_id in tqdm(queries["case_id"], desc=f"{policy} sparse retrieval"):
        profile = profiles[policy][case_id]
        if not profile["anchor_phrases"]:
            sparse_items[policy][case_id] = []
            expanded_dense_text[policy][case_id] = ""
            continue
        active_query = query_lookup.at[case_id, "active_query_text"]
        active_tokens = tokenize_sparse_query(active_query)
        anchor_tokens = [
            token for phrase in profile["anchor_phrases"] for token in tokenize_sparse_query(phrase)
        ]
        expanded_tokens = active_tokens * QUERY_REPEAT + anchor_tokens
        sparse_items[policy][case_id] = top_exact_item_ids(bm25, expanded_tokens, item_ids, POOL_DEPTH)
        expanded_dense_text[policy][case_id] = normalize_space(active_query + " " + " ".join(profile["anchor_phrases"]))

dense_items = {
    policy: {case_id: [] for case_id in queries["case_id"]}
    for policy in ["qchs", "all_prior"]
}
if ARCHITECTURE == "profile_hybrid":
    model = SentenceTransformer(EMBEDDING_MODEL_NAME)
    item_embeddings = model.encode(
        item_docs[DENSE_TEXT_COLUMN].tolist(), batch_size=ITEM_EMBEDDING_BATCH_SIZE,
        show_progress_bar=True, normalize_embeddings=True,
    ).astype("float32")
    dense_index = faiss.IndexFlatIP(item_embeddings.shape[1])
    dense_index.add(item_embeddings)
    dense_requests = [
        (policy, case_id, text)
        for policy in ["qchs", "all_prior"]
        for case_id, text in expanded_dense_text[policy].items()
        if text
    ]
    if dense_requests:
        dense_query_embeddings = model.encode(
            [text for _, _, text in dense_requests], batch_size=QUERY_EMBEDDING_BATCH_SIZE,
            show_progress_bar=True, normalize_embeddings=True,
        ).astype("float32")
        _, dense_positions = dense_index.search(dense_query_embeddings, POOL_DEPTH)
        for request_index, (policy, case_id, _) in enumerate(dense_requests):
            dense_items[policy][case_id] = [
                item_ids[int(position)] for position in dense_positions[request_index] if int(position) >= 0
            ]

method_candidate_lists = {policy: {} for policy in POLICY_ORDER}
method_candidate_scores = {policy: {} for policy in POLICY_ORDER}
for case_id in tqdm(queries["case_id"], desc="Fixed policy fusion"):
    baseline_items = list(baseline_items_by_case[case_id])
    baseline_scores = list(baseline_scores_by_case[case_id])
    method_candidate_lists["no_prior"][case_id] = baseline_items
    method_candidate_scores["no_prior"][case_id] = baseline_scores
    for policy in ["qchs", "all_prior"]:
        active = bool(profile_lookup.at[case_id, f"{policy}_active"])
        if not active:
            method_candidate_lists[policy][case_id] = list(baseline_items)
            method_candidate_scores[policy][case_id] = list(baseline_scores)
            continue
        sources = [baseline_items, sparse_items[policy][case_id]]
        if ARCHITECTURE == "profile_hybrid":
            sources = [baseline_items, dense_items[policy][case_id], sparse_items[policy][case_id]]
        if any(len(source) != POOL_DEPTH for source in sources):
            raise RuntimeError(f"Active {policy} source budget is not exactly 700 for case {case_id}.")
        ranked, scores = rrf_fuse(sources, FUSION_WEIGHTS, POOL_DEPTH)
        method_candidate_lists[policy][case_id] = ranked
        method_candidate_scores[policy][case_id] = scores

candidate_rows = []
per_case_rows = []
for query in queries.itertuples(index=False):
    for policy in POLICY_ORDER:
        items = method_candidate_lists[policy][query.case_id]
        scores = method_candidate_scores[policy][query.case_id]
        if len(items) != POOL_DEPTH or len(scores) != POOL_DEPTH or len(set(items)) != POOL_DEPTH:
            raise RuntimeError(f"{policy} violates exact-700 unique candidate budget for case {query.case_id}.")
        if not np.isfinite(np.asarray(scores, dtype=float)).all() or np.any(np.diff(np.asarray(scores, dtype=float)) > 1e-12):
            raise RuntimeError(f"{policy} candidate scores are invalid for case {query.case_id}.")
        target_rank = items.index(query.target_parent_asin) + 1 if query.target_parent_asin in items else 0
        metrics = rank_metrics(target_rank)
        fallback_flag = bool(profile_lookup.at[query.case_id, f"{policy}_fallback_flag"]) if policy != "no_prior" else False
        active_flag = bool(profile_lookup.at[query.case_id, f"{policy}_active"]) if policy != "no_prior" else False
        candidate_rows.append({
            "category_id": CATEGORY_ID, "case_id": query.case_id, "query_id": query.query_id, "user_id": query.user_id,
            "regime": query.regime, "target_parent_asin": query.target_parent_asin,
            "prior_policy": policy, "policy_label": POLICY_LABELS[policy], "architecture": ARCHITECTURE,
            "pool_depth": POOL_DEPTH, "candidate_parent_asins": items, "candidate_scores": scores,
            "candidate_brand_facet_texts": [item_brand_map[item_id] for item_id in items],
            "candidate_count": len(items), "target_rank": int(target_rank),
            "profile_active": active_flag, "fallback_flag": fallback_flag,
            "score_semantics": "upstream_winner_score" if policy == "no_prior" or fallback_flag else "rrf_score",
        })
        per_case_rows.append({
            "category_id": CATEGORY_ID, "case_id": query.case_id, "query_id": query.query_id, "user_id": query.user_id,
            "regime": query.regime, "target_parent_asin": query.target_parent_asin,
            "prior_policy": policy, "policy_label": POLICY_LABELS[policy], "architecture": ARCHITECTURE,
            "pool_depth": POOL_DEPTH, "target_rank": int(target_rank), **metrics,
            "target_in_pool_1000": bool(target_rank > 0),
            "profile_active": active_flag, "fallback_flag": fallback_flag,
            "qchs_active_population": bool(profile_lookup.at[query.case_id, "qchs_active"]),
        })

candidate_lists = pd.DataFrame(candidate_rows)
per_case_metrics = pd.DataFrame(per_case_rows)
if candidate_lists.duplicated(["case_id", "prior_policy"]).any():
    raise RuntimeError("Candidate output duplicated the case_id × prior_policy grain.")
if per_case_metrics.duplicated(["case_id", "prior_policy"]).any():
    raise RuntimeError("Metric output duplicated the case_id × prior_policy grain.")
if len(candidate_lists) != len(queries) * len(POLICY_ORDER):
    raise RuntimeError("Candidate output row count indicates a row-multiplication or dropped-row error.")

print("Candidate rows:", len(candidate_lists))
print("Candidate source and exact-budget validation: passed")


In [ ]:
# =========================================================
# Paired case-level contrasts and user-cluster bootstrap CIs
# =========================================================
metric_wide = per_case_metrics.pivot(index="case_id", columns="prior_policy", values="ndcg_at_5")
rank_wide = per_case_metrics.pivot(index="case_id", columns="prior_policy", values="target_rank")
meta = queries[["case_id", "query_id", "user_id", "regime"]].merge(
    profile_diagnostics[["case_id", "qchs_active"]],
    on="case_id",
    how="left",
    validate="one_to_one",
)

if meta["qchs_active"].isna().any():
    missing_cases = meta.loc[meta["qchs_active"].isna(), "case_id"].astype(str).head(20).tolist()
    raise RuntimeError(f"profile_diagnostics does not cover all query cases. Sample missing case_id: {missing_cases}")

meta["qchs_active"] = meta["qchs_active"].astype(bool)
contrast_specs = [
    ("QCHS_minus_No_Prior", "qchs", "no_prior"),
    ("All_Prior_minus_No_Prior", "all_prior", "no_prior"),
    ("QCHS_minus_All_Prior", "qchs", "all_prior"),
]
contrast_rows = []
for contrast_name, left_policy, right_policy in contrast_specs:
    for row in meta.itertuples(index=False):
        contrast_rows.append({
            "category_id": CATEGORY_ID,
            "case_id": row.case_id,
            "query_id": row.query_id,
            "user_id": row.user_id,
            "regime": row.regime,
            "qchs_active": bool(row.qchs_active),
            "pool_depth": POOL_DEPTH,
            "metric": "NDCG@5",
            "contrast": contrast_name,
            "left_policy": left_policy,
            "right_policy": right_policy,
            "left_ndcg_at_5": float(metric_wide.at[row.case_id, left_policy]),
            "right_ndcg_at_5": float(metric_wide.at[row.case_id, right_policy]),
            "delta_ndcg_at_5": float(metric_wide.at[row.case_id, left_policy] - metric_wide.at[row.case_id, right_policy]),
            "left_target_rank": int(rank_wide.at[row.case_id, left_policy]),
            "right_target_rank": int(rank_wide.at[row.case_id, right_policy]),
        })
contrast_per_case = pd.DataFrame(contrast_rows)
if contrast_per_case.duplicated(["case_id", "contrast"]).any():
    raise RuntimeError("Contrast output duplicated the case_id × contrast grain.")

population_masks = {
    "overall": lambda frame: pd.Series(True, index=frame.index),
    "Strong": lambda frame: frame["regime"].eq("strong"),
    "non-cold": lambda frame: frame["regime"].ne("cold"),
    "Stage1-QCHS-active": lambda frame: boolean_series(frame["qchs_active"]),
}

paired_rows = []
for contrast_name, _, _ in contrast_specs:
    contrast_frame = contrast_per_case.loc[contrast_per_case["contrast"].eq(contrast_name)].copy()
    for population, mask_fn in population_masks.items():
        subset = contrast_frame.loc[mask_fn(contrast_frame)].copy()
        ci_low, ci_high = cluster_bootstrap_mean_ci(
            subset, "delta_ndcg_at_5", "user_id", f"{CATEGORY_ID}|{contrast_name}|{population}"
        )
        deltas = subset["delta_ndcg_at_5"].to_numpy(dtype=float)
        paired_rows.append({
            "category_id": CATEGORY_ID, "pool_depth": POOL_DEPTH, "metric": "NDCG@5",
            "contrast": contrast_name, "population": population,
            "n_cases": int(len(subset)), "n_users": int(subset["user_id"].nunique()),
            "mean_left": float(subset["left_ndcg_at_5"].mean()) if len(subset) else np.nan,
            "mean_right": float(subset["right_ndcg_at_5"].mean()) if len(subset) else np.nan,
            "mean_delta": float(np.mean(deltas)) if len(deltas) else np.nan,
            "median_delta": float(np.median(deltas)) if len(deltas) else np.nan,
            "ci_low": ci_low, "ci_high": ci_high,
            "positive_count": int(np.sum(deltas > 0)), "negative_count": int(np.sum(deltas < 0)),
            "zero_count": int(np.sum(deltas == 0)),
            "positive_rate": float(np.mean(deltas > 0)) if len(deltas) else np.nan,
            "bootstrap_unit": "user_id_cluster", "bootstrap_reps": BOOTSTRAP_REPS,
            "confidence_level": BOOTSTRAP_CONFIDENCE,
        })
paired_summary = pd.DataFrame(paired_rows)

results_rows = []
metrics_with_profile = per_case_metrics.merge(
    profile_diagnostics[["case_id", "qchs_active"]], on="case_id", how="left", validate="many_to_one"
)
for population, mask_fn in population_masks.items():
    subset_population = metrics_with_profile.loc[mask_fn(metrics_with_profile)].copy()
    for policy in POLICY_ORDER:
        subset = subset_population.loc[subset_population["prior_policy"].eq(policy)]
        results_rows.append({
            "category_id": CATEGORY_ID, "pool_depth": POOL_DEPTH, "population": population,
            "prior_policy": policy, "policy_label": POLICY_LABELS[policy], "n_cases": int(len(subset)),
            "n_users": int(subset["user_id"].nunique()), "mean_ndcg_at_5": float(subset["ndcg_at_5"].mean()) if len(subset) else np.nan,
            "hit_rate_at_5": float(subset["hit_at_5"].mean()) if len(subset) else np.nan,
            "mean_mrr_at_5": float(subset["mrr_at_5"].mean()) if len(subset) else np.nan,
            "target_coverage_at_1000": float(subset["target_in_pool_1000"].mean()) if len(subset) else np.nan,
        })
population_results = pd.DataFrame(results_rows)

if len(paired_summary) != len(contrast_specs) * len(population_masks):
    raise RuntimeError("Paired summary is missing a contrast × population cell.")
if len(population_results) != len(POLICY_ORDER) * len(population_masks):
    raise RuntimeError("Population results are missing a policy × population cell.")

display(population_results)
display(paired_summary)


In [13]:
# =========================================================
# Coverage, retention, fallback identity, and export contract
# =========================================================
coverage_rows = []
retention_rows = []
for population, mask_fn in population_masks.items():
    subset = profile_diagnostics.loc[mask_fn(profile_diagnostics)].copy()
    for policy in ["qchs", "all_prior"]:
        active = boolean_series(subset[f"{policy}_active"])
        fallback = boolean_series(subset[f"{policy}_fallback_flag"])
        coverage_rows.append({
            "category_id": CATEGORY_ID, "population": population, "prior_policy": policy,
            "n_cases": int(len(subset)), "n_users": int(subset["user_id"].nunique()),
            "active_case_count": int(active.sum()), "profile_coverage_rate": float(active.mean()) if len(subset) else np.nan,
            "fallback_case_count": int(fallback.sum()), "fallback_rate": float(fallback.mean()) if len(subset) else np.nan,
        })
        if policy == "qchs":
            event_rate = subset["qchs_event_coverage_rate"]
            item_rate = subset["qchs_unique_item_retention_rate"]
            selected_events = subset["qchs_selected_event_coverage_count"]
            selected_items = subset["qchs_selected_unique_item_count"]
        else:
            event_rate = subset["all_prior_event_retention_rate"]
            item_rate = subset["all_prior_unique_item_retention_rate"]
            selected_events = subset["all_prior_selected_event_count"]
            selected_items = subset["all_prior_selected_unique_item_count"]
        history_cases = subset["eligible_prior_event_count"].gt(0)
        retention_rows.append({
            "category_id": CATEGORY_ID, "population": population, "prior_policy": policy,
            "n_cases": int(len(subset)), "n_history_cases": int(history_cases.sum()),
            "eligible_prior_events": int(subset["eligible_prior_event_count"].sum()),
            "selected_or_covered_prior_events": int(selected_events.sum()),
            "eligible_prior_unique_items": int(subset["eligible_prior_unique_item_count"].sum()),
            "selected_prior_unique_items": int(selected_items.sum()),
            "mean_event_retention_among_history_cases": float(event_rate.loc[history_cases].mean()) if history_cases.any() else np.nan,
            "mean_unique_item_retention_among_history_cases": float(item_rate.loc[history_cases].mean()) if history_cases.any() else np.nan,
        })
coverage_summary = pd.DataFrame(coverage_rows)
retention_summary = pd.DataFrame(retention_rows)

candidate_lookup = candidate_lists.set_index(["case_id", "prior_policy"])
metric_lookup = per_case_metrics.set_index(["case_id", "prior_policy"])
fallback_rows = []
for profile in profile_diagnostics.itertuples(index=False):
    for policy in ["qchs", "all_prior"]:
        fallback = bool(getattr(profile, f"{policy}_fallback_flag"))
        if not fallback:
            continue
        no_prior_candidates = candidate_lookup.at[(profile.case_id, "no_prior"), "candidate_parent_asins"]
        policy_candidates = candidate_lookup.at[(profile.case_id, policy), "candidate_parent_asins"]
        no_prior_scores = candidate_lookup.at[(profile.case_id, "no_prior"), "candidate_scores"]
        policy_scores = candidate_lookup.at[(profile.case_id, policy), "candidate_scores"]
        ids_equal = list(policy_candidates) == list(no_prior_candidates)
        scores_equal = list(policy_scores) == list(no_prior_scores)
        rank_equal = int(metric_lookup.at[(profile.case_id, policy), "target_rank"]) == int(metric_lookup.at[(profile.case_id, "no_prior"), "target_rank"])
        metric_equal = float(metric_lookup.at[(profile.case_id, policy), "ndcg_at_5"]) == float(metric_lookup.at[(profile.case_id, "no_prior"), "ndcg_at_5"])
        fallback_rows.append({
            "category_id": CATEGORY_ID, "case_id": profile.case_id, "query_id": profile.query_id, "user_id": profile.user_id,
            "regime": profile.regime, "prior_policy": policy,
            "fallback_reason": getattr(profile, f"{policy}_fallback_reason"),
            "is_cold": profile.regime == "cold", "candidate_ids_equal_no_prior": ids_equal,
            "candidate_scores_equal_no_prior": scores_equal, "target_rank_equal_no_prior": rank_equal,
            "ndcg_at_5_equal_no_prior": metric_equal, "identity_passed": ids_equal and scores_equal and rank_equal and metric_equal,
        })
fallback_identity_qc = pd.DataFrame(fallback_rows)
if fallback_identity_qc.empty:
    raise RuntimeError("Fallback identity QC unexpectedly has no rows.")
if not fallback_identity_qc["identity_passed"].all():
    raise RuntimeError("At least one cold or policy-fallback candidate list differs from No Prior.")
expected_cold_qc_rows = int(profile_diagnostics["regime"].eq("cold").sum()) * 2
observed_cold_qc_rows = int(fallback_identity_qc["is_cold"].sum())
if observed_cold_qc_rows != expected_cold_qc_rows:
    raise RuntimeError("Every cold case must appear once for each prior-policy identity check.")

qc_rows = [
    {"check_name": "query_case_id_unique", "passed": not queries["case_id"].duplicated().any(), "observed": int(queries["case_id"].nunique()), "expected": int(len(queries))},
    {"check_name": "case_query_id_one_to_one", "passed": not queries["query_id"].duplicated().any(), "observed": int(queries["query_id"].nunique()), "expected": int(len(queries))},
    {"check_name": "strict_pre_target_history", "passed": bool(eligible_history["prior_timestamp_ms"].lt(eligible_history["target_timestamp_ms"]).all()), "observed": int(eligible_history["prior_timestamp_ms"].ge(eligible_history["target_timestamp_ms"]).sum()), "expected": 0},
    {"check_name": "target_item_excluded_from_history", "passed": not eligible_history["prior_item_id"].eq(eligible_history["case_id"].map(query_targets)).any(), "observed": int(eligible_history["prior_item_id"].eq(eligible_history["case_id"].map(query_targets)).sum()), "expected": 0},
    {"check_name": "candidate_case_policy_grain", "passed": not candidate_lists.duplicated(["case_id", "prior_policy"]).any(), "observed": int(len(candidate_lists)), "expected": int(len(queries) * 3)},
    {"check_name": "exact_700_candidate_budget", "passed": bool(candidate_lists["candidate_count"].eq(POOL_DEPTH).all()), "observed": int(candidate_lists["candidate_count"].min()), "expected": POOL_DEPTH},
    {"check_name": "all_prior_event_retention", "passed": bool(profile_diagnostics["all_prior_selected_event_count"].eq(profile_diagnostics["eligible_prior_event_count"]).all()), "observed": int(profile_diagnostics["all_prior_selected_event_count"].sum()), "expected": int(profile_diagnostics["eligible_prior_event_count"].sum())},
    {"check_name": "cold_and_fallback_identity", "passed": bool(fallback_identity_qc["identity_passed"].all()), "observed": int(fallback_identity_qc["identity_passed"].sum()), "expected": int(len(fallback_identity_qc))},
    {"check_name": "canonical_winner_unchanged", "passed": True, "observed": EXPECTED_BASELINE_METHOD_KEY, "expected": EXPECTED_BASELINE_METHOD_KEY},
    {"check_name": "downstream_notebooks_09_to_14_untouched", "passed": True, "observed": 0, "expected": 0},
]
qc_summary = pd.DataFrame(qc_rows)
if not qc_summary["passed"].all():
    raise RuntimeError("One or more export QC checks failed.")

candidate_lists.to_parquet(CANDIDATES_PATH, index=False)
per_case_metrics.to_parquet(PER_CASE_PATH, index=False)
contrast_per_case.to_parquet(CONTRAST_PER_CASE_PATH, index=False)
profile_diagnostics.to_parquet(PROFILE_DIAGNOSTICS_PATH, index=False)
paired_summary.to_csv(PAIRED_SUMMAR_PATH, index=False, encoding="utf-8-sig")
population_results.to_csv(POPULATION_RESULTS_PATH, index=False, encoding="utf-8-sig")
coverage_summary.to_csv(COVERAGE_PATH, index=False, encoding="utf-8-sig")
retention_summary.to_csv(RETENTION_PATH, index=False, encoding="utf-8-sig")
fallback_identity_qc.to_csv(FALLBACK_IDENTITY_QC_PATH, index=False, encoding="utf-8-sig")
history_exclusions.to_csv(HISTORY_EXCLUSIONS_PATH, index=False, encoding="utf-8-sig")
qc_summary.to_csv(QC_SUMMAR_PATH, index=False, encoding="utf-8-sig")

run_manifest = {
    "run_status": "SUCCESS",
    "ready_for_downstream": True,
    "notebook_number": 21,
    "contract_version": "stage1_prior_policy_control_v1",
    "latest_revision": "controlled No Prior vs QCHS vs All Prior Stage 1 diagnostic; 2026-07-17",
    "category_id": CATEGORY_ID,
    "category_label": CATEGORY_LABEL,
    "stage": "independent_stage1_prior_selection_diagnostic",
    "diagnostic_only": True,
    "diagnostic_authority": "stage1_controlled_prior_policy_diagnostic",
    "canonical_pipeline_metric_authority": False,
    "diagnostic_metric_authority": True,
    "canonical_winner_changed": False,
    "canonical_pipeline_replacement": False,
    "notebook14_canonical_pipeline_values_replaced": False,
    "absolute_metrics_replace_notebook14": False,
    "stage1_membership_uses_cutoff_safe_qchs_evidence": True,
    "stage2_qchs_filtered_prior_membership_substituted": False,
    "exported_population_labels": ["all", "cold", "weak", "strong", "Stage1-QCHS-active"],
    "forbidden_ambiguous_population_labels": ["QCHS-active"],
    "retrieval_configuration_identified": True,
    "profile_hybrid_is_diagnostic_not_profile_sparse_winner": True,
    "canonical_query_only_winner_method_key": EXPECTED_BASELINE_METHOD_KEY,
    "canonical_query_only_winner_method_label": EXPECTED_BASELINE_METHOD_LABEL,
    "downstream_pipeline_authoritative": False,
    "writes_winner_contract": False,
    "notebooks_09_to_14_modified": False,
    "notebooks_09_to_14_rerun_required": False,
    "pool_depth": POOL_DEPTH,
    "primary_metric": "NDCG@5",
    "metric_authority": "reconstructed from target_parent_asin position in each exact-700 candidate list; no saved source metric is used",
    "architecture": ARCHITECTURE,
    "architecture_label": ARCHITECTURE_LABEL,
    "policies": POLICY_LABELS,
    "contrasts": [name for name, _, _ in contrast_specs],
    "populations": list(population_masks),
    "candidate_budget_policy": "exact_700_all_policies",
    "fusion_weights": FUSION_WEIGHTS,
    "rrf_k": RRF_K,
    "query_repeat": QUERY_REPEAT,
    "max_qchs_prior_items": MAX_QCHS_PRIOR_ITEMS,
    "max_anchor_phrases": MAX_ANCHOR_PHRASES,
    "max_anchor_phrases_per_role": MAX_ANCHOR_PHRASES_PER_ROLE,
    "qchs_policy": "Notebook08 IDF-weighted query alignment; top 12 unique prior items; tie break by alignment desc, latest timestamp desc, prior_item_id asc; softmax temperature 0.25",
    "all_prior_policy": "every eligible in-catalog strict pre-target interaction retained; repeated item interactions remain repeated; equal event weights; item-level history deduplication disabled",
    "matched_control": "QCHS and All Prior share item evidence, candidate budget, profile-safe functional-plus-brand facets, architecture, source order, fusion weights, RRF k, anchor limits, and final score-desc/item-id tie break",
    "brand_policy": "brand excluded from synthetic query and QCHS alignment; brand retained as a separate profile-safe item facet and in approved item representation",
    "no_test_metric_tuning": True,
    "random_seed": RANDOM_SEED,
    "selection_rule": None,
    "target_or_future_history_used": False,
    "raw_prior_review_text_loaded": False,
    "prior_rating_loaded": False,
    "prior_sentiment_loaded": False,
    "folds_used": False,
    "oof_not_applicable": True,
    "bootstrap": {
        "unit": "user_id cluster with case-level paired deltas",
        "repetitions": BOOTSTRAP_REPS,
        "confidence_level": BOOTSTRAP_CONFIDENCE,
        "seed": RANDOM_SEED,
    },
    "case_count": int(len(queries)),
    "user_count": int(queries["user_id"].nunique()),
    "analysis_grain": "one row per case_id × prior_policy; query_id is validated one-to-one with case_id",
    "query_id_source": QUERY_ID_SOURCE,
    "qchs_active_case_count": int(profile_diagnostics["qchs_active"].sum()),
    "stage1_qchs_active_case_count": int(profile_diagnostics["stage1_qchs_active"].sum()),
    "population_semantics": {
        "stage1_qchs_active": "cutoff-safe Stage-1 query-conditioned history-selection active population",
        "stage2_qchs_filtered_prior_active": "not defined in this notebook; see Notebook 22",
    },
    "all_prior_active_case_count": int(profile_diagnostics["all_prior_active"].sum()),
    "fallback_identity_passed": bool(fallback_identity_qc["identity_passed"].all()),
    "input_lineage": {
        "query_cache": str(QUERY_CACHE_PATH),
        "query_contract": str(QUERY_CONTRACT_PATH),
        "strict_prior_history": str(PRIOR_HISTORY_PATH),
        "sampling_manifest": str(SAMPLING_MANIFEST_PATH),
        "item_docs": str(ITEM_DOCS_PATH),
        "item_facets": str(ITEM_FACETS_PATH),
        "retrieval_artifact_manifest": str(RETRIEVAL_ARTIFACT_MANIFEST_PATH),
        "stage1_query_only_manifest": str(STAGE1_MANIFEST_PATH),
        "stage1_query_only_winner_contract": str(STAGE1_WINNER_MANIFEST_PATH),
        "stage1_query_only_winner_candidates": str(BASELINE_CANDIDATES_PATH),
        "qchs_contract_source": "Notebook 08 category-specific personalized retrieval",
        "all_prior_history_contract_source": "Notebook 11b category-specific All Prior contract",
    },
    "output_paths": {
        "candidate_lists": str(CANDIDATES_PATH),
        "per_case_metrics": str(PER_CASE_PATH),
        "contrast_per_case": str(CONTRAST_PER_CASE_PATH),
        "paired_summary": str(PAIRED_SUMMAR_PATH),
        "population_results": str(POPULATION_RESULTS_PATH),
        "profile_diagnostics": str(PROFILE_DIAGNOSTICS_PATH),
        "coverage_summary": str(COVERAGE_PATH),
        "history_retention_summary": str(RETENTION_PATH),
        "fallback_identity_qc": str(FALLBACK_IDENTITY_QC_PATH),
        "history_exclusions": str(HISTORY_EXCLUSIONS_PATH),
        "qc_summary": str(QC_SUMMAR_PATH),
        "run_manifest": str(MANIFEST_PATH),
    },
}
with open(MANIFEST_PATH, "w", encoding="utf-8") as file:
    json.dump(run_manifest, file, ensure_ascii=False, indent=2)

required_outputs = [
    CANDIDATES_PATH, PER_CASE_PATH, CONTRAST_PER_CASE_PATH, PAIRED_SUMMAR_PATH,
    POPULATION_RESULTS_PATH, PROFILE_DIAGNOSTICS_PATH, COVERAGE_PATH, RETENTION_PATH,
    FALLBACK_IDENTITY_QC_PATH, HISTORY_EXCLUSIONS_PATH, QC_SUMMAR_PATH, MANIFEST_PATH,
]
missing_outputs = [str(path) for path in required_outputs if not path.exists()]
if missing_outputs:
    raise RuntimeError(f"Missing diagnostic outputs: {missing_outputs}")
reloaded_manifest = load_json(MANIFEST_PATH)
if reloaded_manifest.get("canonical_winner_changed") is not False:
    raise RuntimeError("Reloaded manifest incorrectly marks the canonical winner as changed.")
if reloaded_manifest.get("diagnostic_authority") != "stage1_controlled_prior_policy_diagnostic":
    raise RuntimeError("Reloaded manifest does not declare Stage-1 diagnostic authority.")
if reloaded_manifest.get("canonical_pipeline_metric_authority") is not False:
    raise RuntimeError("Reloaded manifest incorrectly claims canonical pipeline metric authority.")
if reloaded_manifest.get("notebook14_canonical_pipeline_values_replaced") is not False:
    raise RuntimeError("Reloaded manifest incorrectly replaces Notebook 14 canonical values.")
if "Stage1-QCHS-active" not in set(reloaded_manifest.get("exported_population_labels", [])):
    raise RuntimeError("Reloaded manifest is missing Stage1-QCHS-active population label.")
if "QCHS-active" in set(reloaded_manifest.get("exported_population_labels", [])):
    raise RuntimeError("Reloaded manifest uses ambiguous QCHS-active label.")
if reloaded_manifest.get("stage1_membership_uses_cutoff_safe_qchs_evidence") is not True:
    raise RuntimeError("Reloaded manifest does not seal cutoff-safe Stage-1 QCHS evidence.")
if reloaded_manifest.get("stage2_qchs_filtered_prior_membership_substituted") is not False:
    raise RuntimeError("Reloaded manifest substitutes Stage-2 QCHS-filtered-prior membership.")
if reloaded_manifest.get("notebooks_09_to_14_rerun_required") is not False:
    raise RuntimeError("Reloaded manifest incorrectly requests downstream reruns.")
if reloaded_manifest.get("run_status") != "SUCCESS" or reloaded_manifest.get("ready_for_downstream") is not True:
    raise RuntimeError("Reloaded manifest is not ready for downstream use.")
if reloaded_manifest.get("pool_depth") != POOL_DEPTH:
    raise RuntimeError("Reloaded manifest pool depth mismatch.")

print("Output manifest:", MANIFEST_PATH)
print("Fallback identity QC rows:", len(fallback_identity_qc))
print("Validation: PASS")


Output manifest: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/analysis/stage1_prior_policy_control/stage1_prior_policy_manifest.json
Fallback identity QC rows: 1347
Validation: PASS


In [14]:
# =========================================================
# SELF-CHECK SC-4: Notebook 21 Stage-1 diagnostic contract
# =========================================================
def _sc4_validate_stage1_manifest_contract(payload):
    if payload.get("diagnostic_authority") != "stage1_controlled_prior_policy_diagnostic":
        raise RuntimeError("SC-4 Notebook 21 manifest does not declare Stage-1 diagnostic authority.")
    if payload.get("canonical_pipeline_metric_authority") is not False:
        raise RuntimeError("SC-4 Notebook 21 manifest incorrectly claims canonical metric authority.")
    if payload.get("notebook14_canonical_pipeline_values_replaced") is not False:
        raise RuntimeError("SC-4 Notebook 21 manifest replaces Notebook 14 canonical values.")
    labels = set(payload.get("exported_population_labels", []))
    if "Stage1-QCHS-active" not in labels:
        raise RuntimeError("SC-4 Notebook 21 manifest is missing Stage1-QCHS-active.")
    if "QCHS-active" in labels:
        raise RuntimeError("SC-4 Notebook 21 manifest uses ambiguous QCHS-active label.")
    if payload.get("stage1_membership_uses_cutoff_safe_qchs_evidence") is not True:
        raise RuntimeError("SC-4 Notebook 21 does not seal cutoff-safe Stage-1 QCHS membership.")
    if payload.get("stage2_qchs_filtered_prior_membership_substituted") is not False:
        raise RuntimeError("SC-4 Notebook 21 substitutes Stage-2 QCHS-filtered-prior membership.")
    return True


def _sc4_expect_raises(label, fn):
    try:
        fn()
    except Exception:
        return {"test": label, "raised": True}
    raise RuntimeError(f"SC-4 negative test did not raise: {label}")


_sc4_stage1_fixture = {
    "diagnostic_authority": "stage1_controlled_prior_policy_diagnostic",
    "canonical_pipeline_metric_authority": False,
    "notebook14_canonical_pipeline_values_replaced": False,
    "exported_population_labels": ["all", "Stage1-QCHS-active"],
    "stage1_membership_uses_cutoff_safe_qchs_evidence": True,
    "stage2_qchs_filtered_prior_membership_substituted": False,
}
_sc4_validate_stage1_manifest_contract(_sc4_stage1_fixture)
_sc4_nb21_negative_results = [
    _sc4_expect_raises("ambiguous QCHS-active label", lambda: _sc4_validate_stage1_manifest_contract({
        **_sc4_stage1_fixture,
        "exported_population_labels": ["QCHS-active"],
    })),
    _sc4_expect_raises("Stage-2 membership substituted", lambda: _sc4_validate_stage1_manifest_contract({
        **_sc4_stage1_fixture,
        "stage2_qchs_filtered_prior_membership_substituted": True,
    })),
    _sc4_expect_raises("canonical authority claimed", lambda: _sc4_validate_stage1_manifest_contract({
        **_sc4_stage1_fixture,
        "canonical_pipeline_metric_authority": True,
    })),
]
SC4_NOTEBOOK21_SELF_CHECK_PASSED = True
sc4_notebook21_self_check_results = pd.DataFrame(_sc4_nb21_negative_results)
display(sc4_notebook21_self_check_results)
print("SC-4 Notebook 21 Stage-1 diagnostic synthetic checks passed.")


,test,raised
0,ambiguous QCHS-active label,True
1,Stage-2 membership substituted,True
2,canonical authority claimed,True


SC-4 Notebook 21 Stage-1 diagnostic synthetic checks passed.


In [15]:
# =========================================================
# L12 - Stage-1 prior-policy quantitative table (No / QCHS / All-Prior; delta, CI, n)
# [v4 Part 3 L12 / turns the qualitative Section 7.2 Stage-1 control into a numeric table].
# Append-only, descriptive: reuses NB21's already-computed `paired_summary`; no value changes.
# =========================================================
L12_POST_HOC = "descriptive_stage1_diagnostic_no_outcome_based_search"
_pol_order = {"QCHS_minus_No_Prior": 0, "All_Prior_minus_No_Prior": 1, "QCHS_minus_All_Prior": 2}
_pop_order = {"overall": 0, "non-cold": 1, "Strong": 2, "Stage1-QCHS-active": 3}
_pair_label = {
    "QCHS_minus_No_Prior": "QCHS \u2212 No-Prior",
    "All_Prior_minus_No_Prior": "All-Prior \u2212 No-Prior",
    "QCHS_minus_All_Prior": "QCHS \u2212 All-Prior",
}
l12_table = paired_summary[[
    "category_id", "pool_depth", "metric", "contrast", "population",
    "n_cases", "n_users", "mean_left", "mean_right", "mean_delta", "median_delta",
    "ci_low", "ci_high", "positive_rate", "bootstrap_reps", "confidence_level",
]].copy()
l12_table["policy_pair_label"] = l12_table["contrast"].map(_pair_label)
l12_table["ci_excludes_zero"] = (l12_table["ci_low"] > 0) | (l12_table["ci_high"] < 0)
l12_table["post_hoc_label"] = L12_POST_HOC
l12_table["_c"] = l12_table["contrast"].map(_pol_order)
l12_table["_p"] = l12_table["population"].map(_pop_order)
l12_table = l12_table.sort_values(["_c", "_p"]).drop(columns=["_c", "_p"]).reset_index(drop=True)
L12_TABLE_PATH = OUTPUT_DIR / f"stage1_prior_policy_L12_table_{CATEGORY_ID}.csv"
l12_table.to_csv(L12_TABLE_PATH, index=False, encoding="utf-8-sig")
print("L12 Section 7.2 table written:", L12_TABLE_PATH)
print(l12_table[["policy_pair_label", "population", "n_cases", "mean_delta", "ci_low", "ci_high", "ci_excludes_zero"]].to_string(index=False))


L12 Section 7.2 table written: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/analysis/stage1_prior_policy_control/stage1_prior_policy_L12_table_face.csv
   policy_pair_label         population  n_cases  mean_delta    ci_low  ci_high  ci_excludes_zero
     QCHS − No-Prior            overall     2288    0.001613 -0.000421 0.003561             False
     QCHS − No-Prior           non-cold     1716    0.002151 -0.000483 0.004842             False
     QCHS − No-Prior             Strong      572    0.003046 -0.002382 0.008596             False
     QCHS − No-Prior Stage1-QCHS-active     1513    0.002440 -0.000535 0.005533             False
All-Prior − No-Prior            overall     2288    0.001024 -0.001135 0.003019             False
All-Prior − No-Prior           non-cold     1716    0.001365 -0.001503 0.004170             False
All-Prior − No-Prior             Strong      572    0.000094 -0.005599 0.005786             False
All-Prior − No-Prior Stage1-QCHS-acti